# Module 3 — Lab D: MLOps — Hyperparameter Tuning & Best-Model Selection
## Beat the Lab C baseline (XGBoost F1 = 0.679) with AutoML + Optuna search

---

### Why a separate lab?

Lab C established a **baseline**: three classifiers trained with reasonable-but-untuned hyperparameters. XGBoost won with **F1 = 0.679** on the validation set and **F1 ≈ 0.68** on the held-out test set. The operations team at FreshBasket is asking: *can we push the F1 higher with the same features?*

This lab follows the same "compare → tune → finalize → honest score" lifecycle as the Module 2 PyCaret benchmark (`pycaret_benchmark_v3.py`), adapted for classification:

| Step | What | Why |
|---|---|---|
| 1 | Load `final_features.csv` + recreate **Lab C's exact splits** | apples-to-apples comparison |
| 2 | PyCaret `setup()` + `compare_models()` — sweep ~15 classifiers | wide net for the search |
| 3 | `tune_model(n_iter=50)` on the top performer | deepen the search on the winner |
| 4 | `blend_models()` + `stack_models()` for ensemble experiments | sometimes a blend > the best single model |
| 5 | **Score on Lab C's held-out test set ONCE** | honest, no test-set contamination |
| 6 | If F1 > 0.679 → register a **new version** in the existing MLflow registry under `truck-delay-classifier` | downstream apps pick up the improvement automatically |
| 7 | Save tuned model + preprocessing pipeline to S3 | Lab D can switch to it |

### The discipline rule (carried over from v3)

After Step 5, **no decisions** are made based on the test number. If the tuned model disappoints on the held-out test, we do NOT re-tune or pick a different ensemble — that would contaminate the test set. We report the number honestly and either ship or keep the Lab C XGBoost.

### New MLflow experiment

All runs from this notebook log to a **separate** experiment so the Lab C runs stay clean:

```
Experiment 1 (Lab C):       'truck-delay-classification'    ← baseline runs
Experiment 2 (this lab):    'truck-delay-hp-tuning'         ← ~70+ tuning runs
```

The model registry name stays the same (`truck-delay-classifier`), so a successful tune simply bumps the version number from v1 → v2.

### Artifact chain

```
Lab B  →  final_features.csv (S3)                                           ─┐
Lab C  →  v1 of 'truck-delay-classifier' (XGBoost, F1≈0.68)                   │
                                                                              ▼
Lab D MLOps (this notebook)
       →  ~70 MLflow runs in 'truck-delay-hp-tuning'
       →  IF winner beats baseline: v2 of 'truck-delay-classifier' + S3 .pkl
       →  ELSE: comparison table + decision to keep v1

Lab D (downstream)
       →  picks up whatever's in the Staging slot of 'truck-delay-classifier'
```

### Estimated run-time on SageMaker (ml.t3.medium)

| Phase | Time |
|---|---|
| PyCaret first-install (`!pip install pycaret -q`) | 3–5 min |
| `compare_models` (~15 algorithms × 5-fold CV) | 5–8 min |
| `tune_model(n_iter=50)` | 3–5 min |
| `blend_models` + `stack_models` | 2–3 min |
| Honest test scoring + registration | <1 min |
| **Total (first run)** | **~15–25 min** |
| Subsequent runs (PyCaret already installed) | ~12–20 min |


---

## 1. Environment Setup

Two cells: install + import. PyCaret bundles XGBoost, LightGBM, CatBoost, sklearn, and ~30 other deps — the first-run install on SageMaker is the slow step.

> **SageMaker:** uncomment the `pip install` line once. After that PyCaret stays in the conda env.


In [ ]:
# -- Uncomment for first run on SageMaker / Colab / local --
# !pip install -q pycaret==3.3.2 optuna mlflow boto3

# -- Imports --
import os, sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import boto3
import joblib
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")

print("Core imports OK")
print(f"  pandas      {pd.__version__}")
print(f"  numpy       {np.__version__}")
print(f"  mlflow      {mlflow.__version__}")


**MLflow configuration — new experiment name.** Same tracking server (EC2), same registry, but a **different experiment** keeps the ~70 tuning runs from cluttering Lab C's baseline experiment.


In [ ]:
# ================================================================
#  MLflow Configuration — NEW experiment for HP tuning
# ================================================================
MLFLOW_TRACKING_URI = "http://<EC2_PUBLIC_IP>:5000"        # <-- EDIT (same as Lab C)
EXPERIMENT_NAME     = "truck-delay-hp-tuning"               # <-- NEW: separate from Lab C
REGISTERED_MODEL_NAME = "truck-delay-classifier"            # same name — bumps version

# -- S3 Configuration --
S3_BUCKET    = "mlops-m3-batch-2026-<AWS_ACCOUNT_ID>"       # <-- EDIT if different
S3_DATA_KEY  = "data/processed/final_features.csv"
S3_MODEL_DIR = "models/truck-delay-tuned/"

# -- Connect to MLflow --
try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT_NAME)
    mlflow.search_experiments()
    print(f"Connected to MLflow: {MLFLOW_TRACKING_URI}")
    print(f"Experiment:         '{EXPERIMENT_NAME}'  (NEW — separate from Lab C)")
    print(f"Registry target:    '{REGISTERED_MODEL_NAME}'  (will bump version on a win)")
    MLFLOW_AVAILABLE = True
except Exception as e:
    print(f"Could not reach MLflow: {e}")
    print("Falling back to local ./mlruns")
    mlflow.set_tracking_uri("mlruns")
    mlflow.set_experiment(EXPERIMENT_NAME)
    MLFLOW_AVAILABLE = False


---

## 2. Load Feature Matrix + Recreate Lab C's Splits

To compare against the F1 = 0.679 baseline **fairly**, we must use the same train/val/test split Lab C used: stratified 70/15/15 with `random_state=42`. Anything else and the comparison is contaminated.


In [ ]:
# -- Load final_features.csv from S3 with local fallback --
LOCAL_PATH = "data/processed/final_features.csv"
Path("data/processed").mkdir(parents=True, exist_ok=True)

try:
    s3 = boto3.client("s3")
    s3.download_file(S3_BUCKET, S3_DATA_KEY, LOCAL_PATH)
    print(f"Downloaded from S3: s3://{S3_BUCKET}/{S3_DATA_KEY}")
except Exception as e:
    print(f"S3 download failed: {e}")
    if not os.path.exists(LOCAL_PATH):
        raise FileNotFoundError(f"{LOCAL_PATH} not found locally either.")

df = pd.read_csv(LOCAL_PATH)
print(f"\nLoaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Target distribution:")
print(df["delay"].value_counts())


In [ ]:
# ================================================================
#  Recreate Lab C's exact splits (stratified 70/15/15, random_state=42)
# ================================================================
TARGET = "delay"

# -- Drop ID-like columns that aren't features --
DROP_COLS = [c for c in ["truck_id", "route_id", "driver_id", "departure_date",
                          "estimated_arrival", "origin_id", "destination_id"]
             if c in df.columns]
X_all = df.drop(columns=DROP_COLS + [TARGET])
y_all = df[TARGET]

# -- Stratified 70 / 15 / 15 (matches Lab C) --
X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y_all, test_size=0.15, random_state=42, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp
)   # 0.1765 ≈ 15/85 → final 70/15/15

print(f"Train: {X_train.shape}  ({y_train.mean():.3f} delay rate)")
print(f"Val:   {X_val.shape}    ({y_val.mean():.3f} delay rate)")
print(f"Test:  {X_test.shape}    ({y_test.mean():.3f} delay rate)")
print(f"\nFeatures: {X_train.shape[1]}")

# -- Baseline to beat --
BASELINE_F1   = 0.679
BASELINE_NAME = "Lab C XGBoost (untuned)"
print(f"\nBaseline to beat:  {BASELINE_NAME}  →  F1 = {BASELINE_F1}")


---

## 3. PyCaret Setup + Wide Sweep (compare_models)

`pycaret.classification.setup()` is PyCaret's preprocessing + experiment-tracking entry point. We pass the **training pool only** (train + val combined, since PyCaret does its own 5-fold CV); the **test set stays untouched** until Step 5.

Key settings:
- `target="delay"` — binary classification
- `train_size=0.85` — 85% of what we pass = training pool; 15% = PyCaret's internal holdout (we won't use it because we have our own test set)
- `fold=5` — 5-fold CV for the leaderboard (faster than 10 on 10K+ rows)
- `log_experiment=True` — every model PyCaret trains gets logged to MLflow automatically under our `truck-delay-hp-tuning` experiment
- `experiment_name=EXPERIMENT_NAME` — match what we set above


In [ ]:
# -- Apply the v3 reference's MLflow-compat patch (PyCaret 3.x + MLflow 2.x+) --
def _patch_pycaret_mlflow_compat():
    from contextlib import contextmanager
    import pycaret.loggers.mlflow_logger as ml
    ml._active_run_stack = []

    @contextmanager
    def _noop():
        yield

    ml.clean_active_mlflow_run = _noop


def _silence_pycaret_noise():
    import logging
    logging.getLogger("mlflow").setLevel(logging.ERROR)
    try:
        import lightgbm as lgb

        class _SilentLogger:
            def info(self, msg): pass
            def warning(self, msg): pass

        lgb.register_logger(_SilentLogger())
    except ImportError:
        pass


from pycaret.classification import (
    setup, compare_models, tune_model, blend_models, stack_models,
    finalize_model, predict_model, pull, save_model, plot_model,
)

_patch_pycaret_mlflow_compat()
_silence_pycaret_noise()

# -- PyCaret expects the target as a column in a single dataframe --
# Combine train + val for PyCaret's CV pool (test stays out)
pycaret_pool = pd.concat([X_train, X_val], axis=0).copy()
pycaret_pool[TARGET] = pd.concat([y_train, y_val], axis=0).values

print(f"PyCaret pool: {pycaret_pool.shape}  (train + val, test held out)")


In [ ]:
# ================================================================
#  PyCaret setup() — preprocessing + experiment tracking
# ================================================================
exp = setup(
    data=pycaret_pool,
    target=TARGET,
    session_id=42,
    train_size=0.85,                   # PyCaret holds out 15% internally; we ignore it
    normalize=True,                    # StandardScaler equivalent (matches Lab C)
    transformation=False,
    fold=5,                            # 5-fold CV — speed/fidelity sweet spot here
    fold_strategy="stratifiedkfold",
    fix_imbalance=False,               # 60/40 delay rate isn't bad enough to need SMOTE
    log_experiment=True,
    experiment_name=EXPERIMENT_NAME,   # all runs go to truck-delay-hp-tuning
    log_plots=False,                   # skipping plots speeds up logging
    verbose=False,
)

print(f"\nPyCaret setup complete. Experiment: {EXPERIMENT_NAME}")
print("All subsequent runs auto-log to MLflow.")


**`compare_models`** trains every algorithm in PyCaret's classification module on the training pool, runs 5-fold CV, and returns a leaderboard sorted by the metric of choice. We sort by **F1** because that's the business metric (matches Lab C's choice).

Expected: ~15 algorithms × 5 folds = ~75 model fits. Takes 5-8 min on SageMaker.


In [ ]:
# ================================================================
#  compare_models — sweep ~15 classifiers, sort by F1
# ================================================================
print("Running compare_models(n_select=5, sort='F1')...")
print("(This will take 5-8 min and log every model to MLflow)\n")

top5 = compare_models(
    n_select=5,
    sort="F1",
    turbo=True,        # skip a few slow ones (GP, MLP); their gain rarely justifies the time
    verbose=True,
)

leaderboard = pull()
print("\n=== Leaderboard (top 10 by F1) ===")
print(leaderboard.head(10).to_string())


---

## 4. Tune the Top Performer

`tune_model(n_iter=50)` runs a randomized search over PyCaret's pre-defined hyperparameter grid for the winner from `compare_models`. We use 50 iterations as a reasonable budget; bump to 100+ if you have time.

`choose_better=True` means PyCaret only returns the tuned model if it actually beat the default — otherwise it keeps the default. This prevents the tuning from making things worse.


In [ ]:
# ================================================================
#  Tune the top model — randomized search, 50 iterations
# ================================================================
best_model = top5[0]
best_name  = type(best_model).__name__
print(f"Tuning {best_name} (n_iter=50)...")

tuned_top = tune_model(
    best_model,
    n_iter=50,
    optimize="F1",
    search_library="scikit-learn",   # try "optuna" too — TPE search, usually 5-15% better
    search_algorithm="random",
    choose_better=True,              # only return tuned if it beats default
    verbose=False,
)

tune_results = pull()
print("\n=== Tuning CV results (5-fold) ===")
print(tune_results.to_string())


---

## 5. Ensembles — Blend & Stack

Sometimes a blend of the top performers beats the best single model. Two strategies:

- **`blend_models`** — soft voting (average predicted probabilities). Robust, simple.
- **`stack_models`** — train a meta-learner (logistic regression by default) on the predictions of the base models. Higher ceiling, higher risk of overfitting.

Both get logged to MLflow as separate runs.


In [ ]:
# ================================================================
#  Blend the top 3 models (soft voting)
# ================================================================
print("Blending top 3 models...")
blended = blend_models(estimator_list=top5[:3], method="soft", verbose=False)
blend_cv = pull()
print("\n=== Blend CV results ===")
print(blend_cv.to_string())


In [ ]:
# ================================================================
#  Stack the top 3 models (meta-learner = logistic regression)
# ================================================================
print("Stacking top 3 models...")
stacked = stack_models(estimator_list=top5[:3], meta_model=None, verbose=False)
stack_cv = pull()
print("\n=== Stack CV results ===")
print(stack_cv.to_string())


---

## 6. Candidate Showdown on the Validation Set

Four candidates so far:
1. Top model from `compare_models` (un-tuned)
2. Tuned top model (`tuned_top`)
3. Blended top 3 (`blended`)
4. Stacked top 3 (`stacked`)

We pick the winner on the **validation set** — NOT the test set. The test set is still untouched and stays that way until Section 7.


In [ ]:
# ================================================================
#  Helper: score a PyCaret pipeline on (X, y) → metrics dict
# ================================================================
def score_pycaret(model, X, y, label):
    pred_df = predict_model(model, data=X, verbose=False)
    # PyCaret 3.x: 'prediction_label' for class, 'prediction_score' for proba
    y_pred = pred_df["prediction_label"].values
    y_prob = pred_df.get("prediction_score", pd.Series(np.zeros(len(y)))).values
    return {
        "name":      label,
        "accuracy":  accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred),
        "recall":    recall_score(y, y_pred),
        "f1":        f1_score(y, y_pred),
        "roc_auc":   roc_auc_score(y, y_prob) if len(np.unique(y_prob)) > 1 else np.nan,
    }


# -- Score all 4 candidates on the validation set --
candidates = [
    (top5[0],   f"Top-1 untuned ({best_name})"),
    (tuned_top, f"Top-1 tuned ({best_name})"),
    (blended,   "Blend (top-3 soft vote)"),
    (stacked,   "Stack (top-3, LR meta)"),
]

val_results = [score_pycaret(m, X_val, y_val, n) for m, n in candidates]
val_df = pd.DataFrame(val_results).set_index("name").round(4)
print("=== Validation-set comparison ===")
print(val_df.to_string())

# -- Pick the winner (highest F1 on val) --
winner_name   = val_df["f1"].idxmax()
winner_model  = [m for m, n in candidates if n == winner_name][0]
winner_val_f1 = val_df.loc[winner_name, "f1"]
print(f"\nValidation winner: {winner_name}  (F1 = {winner_val_f1:.4f})")


---

## 7. Honest Test-Set Evaluation — One Shot, No Going Back

This is the **moment of truth**. We score the validation winner on the held-out test set **exactly once**. Whatever F1 we get is the number that goes in the registry / on the slide / in the email to Priya.

If the tuned model beats `BASELINE_F1 = 0.679` → bump the registry version.

If it doesn't → we report the result honestly, do NOT keep tuning (that would contaminate the test), and keep the Lab C XGBoost v1 in production.


In [ ]:
# ================================================================
#  Final test-set evaluation — runs ONCE
# ================================================================
print("Final scoring on held-out test set (the one Lab C also used)...\n")

final_score = score_pycaret(winner_model, X_test, y_test, winner_name)
final_df = pd.DataFrame([final_score]).set_index("name").round(4)
print("=== TEST SET (final, honest) ===")
print(final_df.to_string())

# -- Compare to Lab C baseline --
delta_f1       = final_score["f1"] - BASELINE_F1
beats_baseline = final_score["f1"] > BASELINE_F1

bar = "=" * 60
print(f"\n{bar}")
print(f"  Lab C baseline F1:  {BASELINE_F1:.4f}  ({BASELINE_NAME})")
print(f"  Tuned model F1:     {final_score['f1']:.4f}  ({winner_name})")
print(f"  Delta:              {delta_f1:+.4f}")
verdict = "YES — register new version" if beats_baseline else "NO — keep Lab C v1 in production"
print(f"  Beats baseline?     {verdict}")
print(bar)


In [ ]:
# ================================================================
#  Confusion matrix + classification report for the winner
# ================================================================
pred_df = predict_model(winner_model, data=X_test, verbose=False)
y_pred_test = pred_df["prediction_label"].values

print("=== Classification Report (test set) ===")
print(classification_report(y_test, y_pred_test,
                            target_names=["On-Time", "Delayed"]))

cm = confusion_matrix(y_test, y_pred_test)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["On-Time", "Delayed"],
            yticklabels=["On-Time", "Delayed"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"{winner_name} — Test Confusion Matrix")
plt.tight_layout()
plt.show()


---

## 8. Finalize + Register (only if we beat the baseline)

Following the v3 discipline: after the test number is locked in, we **refit the same model on (train + val + test) combined** for the deployment artifact. The reasoning is identical to v3:

> The test set has done its one job — it gave us an honest performance estimate. We don't need to hold it out anymore. The deployed model should learn from every row of signal we have.

Then register the new version in `truck-delay-classifier` (same registry name as Lab C).


In [ ]:
# ================================================================
#  Finalize on ALL data + save artifacts + register
# ================================================================
if not beats_baseline:
    print("Tuned model did not beat the baseline — skipping registration.")
    print(f"Lab C v1 ({BASELINE_NAME}) remains the production model.")
    print(f"\nThis lab still produced ~70+ MLflow runs in '{EXPERIMENT_NAME}'")
    print("for the team to inspect and learn from.")
else:
    print(f"Finalizing winner on FULL data (train + val + test = {len(X_all)} rows)...")

    full_pool = X_all.copy()
    full_pool[TARGET] = y_all.values

    # Re-run setup on full data so the preprocessing pipeline reflects everything
    # the deployed model will see in production.
    setup(
        data=full_pool,
        target=TARGET,
        session_id=42,
        train_size=0.99,                  # PyCaret needs SOME holdout — sliver, unused
        normalize=True,
        fold=3,
        log_experiment=False,             # we're not making decisions on this fit
        verbose=False,
    )

    deployment_model = finalize_model(winner_model)
    print(f"Deployment model: {type(deployment_model).__name__}")

    # -- Save locally --
    Path("artifacts").mkdir(exist_ok=True)
    audit_path  = "artifacts/truck_delay_tuned_AUDIT"    # scored on test
    deploy_path = "artifacts/truck_delay_tuned_DEPLOY"   # refit on full
    save_model(winner_model,      audit_path)
    save_model(deployment_model,  deploy_path)
    print(f"  Audit copy:      {audit_path}.pkl  ({len(pycaret_pool)} rows, scored)")
    print(f"  Deployment copy: {deploy_path}.pkl  ({len(X_all)} rows, NOT scored)")


In [ ]:
# ================================================================
#  Register the deployment artifact in MLflow Model Registry
# ================================================================
if beats_baseline:
    run_name = f"WINNER_{winner_name}_test_f1_{final_score['f1']:.4f}"
    with mlflow.start_run(run_name=run_name):
        # -- Tags --
        mlflow.set_tag("project", "truck-delay-classification")
        mlflow.set_tag("module", "M3")
        mlflow.set_tag("phase", "hp-tuning")
        mlflow.set_tag("model_type", winner_name)
        mlflow.set_tag("evaluation", "final_test_set")
        mlflow.set_tag("beats_baseline", "true")

        # -- Params --
        mlflow.log_param("winner_model", winner_name)
        mlflow.log_param("baseline_f1", BASELINE_F1)
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_val",   len(X_val))
        mlflow.log_param("n_test",  len(X_test))
        mlflow.log_param("n_deploy_refit", len(X_all))

        # -- Metrics --
        for k, v in final_score.items():
            if k != "name" and isinstance(v, (int, float)) and not np.isnan(v):
                mlflow.log_metric(f"test_{k}", v)
        mlflow.log_metric("test_f1_delta_vs_baseline", delta_f1)

        # -- Log + register the deployment model as a NEW VERSION --
        # PyCaret pipelines pickle as sklearn objects — log via mlflow.sklearn
        mlflow.sklearn.log_model(
            deployment_model,
            name="model",
            registered_model_name=REGISTERED_MODEL_NAME,   # bumps version
        )

        run_id = mlflow.active_run().info.run_id
        print(f"\nRegistered new version of '{REGISTERED_MODEL_NAME}'")
        print(f"  Run ID:   {run_id}")
        print(f"  Test F1:  {final_score['f1']:.4f}  "
              f"(baseline {BASELINE_F1:.4f}, +{delta_f1:.4f})")
        print(f"  View at:  {MLFLOW_TRACKING_URI}/#/models/{REGISTERED_MODEL_NAME}")

    # -- Transition to Staging --
    try:
        client = mlflow.tracking.MlflowClient()
        latest = client.get_latest_versions(REGISTERED_MODEL_NAME, stages=["None"])[0]
        client.transition_model_version_stage(
            name=REGISTERED_MODEL_NAME,
            version=latest.version,
            stage="Staging",
            archive_existing_versions=False,    # keep v1 around for rollback
        )
        print(f"  Transitioned {REGISTERED_MODEL_NAME} v{latest.version} → Staging")
        print("  (v1 from Lab C is preserved — you can roll back if needed)")
    except Exception as e:
        print(f"Stage transition failed (non-fatal): {e}")
else:
    print("Skipping registration — tuned model didn't beat the baseline.")


---

## 9. Upload Tuned Artifacts to S3 (for Lab D)

If we have a winner, push the .pkl to S3 so Lab D's `utils.load_artifacts()` can pick it up. The path matches Lab C's S3 layout but under a separate prefix (`models/truck-delay-tuned/`) so the Lab C v1 artifact isn't overwritten.


In [ ]:
# ================================================================
#  Upload deployment artifact to S3
# ================================================================
if beats_baseline:
    try:
        s3 = boto3.client("s3")
        local_pkl = f"{deploy_path}.pkl"
        s3_key = f"{S3_MODEL_DIR}tuned_pipeline.pkl"
        s3.upload_file(local_pkl, S3_BUCKET, s3_key)
        print(f"Uploaded: s3://{S3_BUCKET}/{s3_key}")

        # Also upload a small metadata JSON for traceability
        meta = {
            "winner_model": winner_name,
            "test_f1":       float(final_score["f1"]),
            "test_accuracy": float(final_score["accuracy"]),
            "test_precision":float(final_score["precision"]),
            "test_recall":   float(final_score["recall"]),
            "test_roc_auc":  float(final_score["roc_auc"]) if not np.isnan(final_score["roc_auc"]) else None,
            "baseline_f1":   BASELINE_F1,
            "delta_vs_baseline": float(delta_f1),
            "n_train_pool":  len(X_train) + len(X_val),
            "n_test":        len(X_test),
            "n_deploy_refit":len(X_all),
            "mlflow_experiment": EXPERIMENT_NAME,
            "registered_model":  REGISTERED_MODEL_NAME,
        }
        meta_path = "artifacts/tuned_metadata.json"
        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)
        s3.upload_file(meta_path, S3_BUCKET, f"{S3_MODEL_DIR}tuned_metadata.json")
        print(f"Uploaded: s3://{S3_BUCKET}/{S3_MODEL_DIR}tuned_metadata.json")
    except Exception as e:
        print(f"S3 upload failed (non-fatal — model is still in MLflow): {e}")
else:
    print("Skipped S3 upload — no improvement to ship.")


---

## 10. Summary & Next Steps

What this lab produced:

1. **~70+ MLflow runs** in the `truck-delay-hp-tuning` experiment — every algorithm PyCaret tried, every tuning iteration, plus the blend / stack / final winner.
2. **A head-to-head F1 comparison** of 4 candidate strategies (untuned top-1, tuned top-1, blend, stack).
3. **One honest test-set number** for the validation winner — what gets reported to Priya.
4. **(Conditionally)** a new version of `truck-delay-classifier` in the MLflow Model Registry, in `Staging`. Lab E Streamlit picks this up automatically if you point it at the Staging slot.

**What did NOT happen** (by design):
- No re-tuning after the test score landed.
- No swapping algorithms post-hoc to chase a number.
- No deletion of the Lab C v1 — it stays as a rollback target.

### Where to look in the MLflow UI

```
http://<EC2_PUBLIC_IP>:5000

  Experiments
    ├── truck-delay-classification    ← Lab C baseline runs (LR, RF, XGBoost)
    └── truck-delay-hp-tuning         ← THIS LAB — ~70 runs to scroll through

  Models
    └── truck-delay-classifier
          ├── Version 1   (Lab C XGBoost, F1≈0.68)
          └── Version 2   (Lab D MLOps winner — Staging, if it beat baseline)
```

### Caveats

- **PyCaret's blend/stack outputs are sklearn pipelines** — they pickle cleanly and Lab D's `utils.apply_prediction_pipeline()` should consume them as-is, BUT the column order matters. Verify `feature_names_in_` on the loaded model matches what Lab D's `fetch_predictions_data` returns.
- **The "F1 = 0.679 baseline"** is from your specific Lab C run. Re-run Lab C if you want an updated baseline — small dataset noise can shift this by ±0.01.
- **If you switch `search_library` to `"optuna"`** in `tune_model`, you get TPE-based search instead of random — usually 5–15% better, ~30% slower. Worth it on production runs.

### If you want to go deeper

- Bump `n_iter=50 → 200` in `tune_model`.
- Add `class_weight` tuning to Logistic Regression and Random Forest (could help precision/recall trade-off).
- Try **Optuna directly** (without PyCaret) for fine-grained control over the search space — especially for XGBoost where the default PyCaret grid is conservative.
